### Generación automática de QA

Lo haremos de forma controlada: no vamos a mandar todos los fragmentos a un único prompt genérico. El script va a:

Leer fragmentos_semanticos.jsonl.
Clasificar cada fragmento por tipo.
Seleccionar una plantilla de generación según el tipo.
Generar varias preguntas por fragmento.
Mantener la respuesta basada exclusivamente en el texto fuente.
Conservar fragmento_id, categoría, URL y documento.
Eliminar QA duplicados.
Detectar respuestas demasiado cortas o sospechosas.
Guardar un dataset intermedio para revisión.
Finalmente producir train.jsonl y validation.jsonl.


### 1. Estructura

Al finalizar tendremos:

UniversidadLLM/
│
├── semantico/
│   └── fragmentos_semanticos.jsonl
│
└── dataset/
    ├── qa_generados.jsonl
    ├── qa_filtrados.jsonl
    ├── qa_revision.csv
    ├── train.jsonl
    └── validation.jsonl

### 2. Instalar librerías

En Colab:

In [ ]:
!pip install -q openai pandas tqdm

### 3. Configuración

In [ ]:
import os
import json
import re
import time
import random
import hashlib

import pandas as pd

from tqdm.auto import tqdm
from openai import OpenAI

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
BASE_DIR = "/content/drive/MyDrive/UniversidadLLM"

SEMANTIC_FILE = os.path.join(
    BASE_DIR,
    "semantico",
    "fragmentos_semanticos.jsonl"
)

DATASET_DIR = os.path.join(
    BASE_DIR,
    "dataset"
)

os.makedirs(
    DATASET_DIR,
    exist_ok=True
)

print(SEMANTIC_FILE)

/content/drive/MyDrive/UniversidadLLM/semantico/fragmentos_semanticos.jsonl


In [6]:
from google.colab import userdata

OPENAI_API_KEY = userdata.get(
    "OPENAI_API_KEY"
)

client = OpenAI(
    api_key=OPENAI_API_KEY
)

### 5. Modelo para generar el dataset

In [7]:
GENERATION_MODEL = "gpt-4.1-mini"

### 6. Cargar los fragmentos

In [8]:
def cargar_jsonl(path):

    registros = []

    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        for linea in f:

            linea = linea.strip()

            if not linea:
                continue

            registros.append(
                json.loads(linea)
            )

    return registros


fragmentos = cargar_jsonl(
    SEMANTIC_FILE
)

print(
    f"Fragmentos cargados: {len(fragmentos)}"
)

Fragmentos cargados: 1043


### 7. Revisar distribución

In [9]:
df_fragmentos = pd.DataFrame(
    fragmentos
)

df_fragmentos["tipo"].value_counts()

# Tambien
df_fragmentos["categoria"].value_counts()
# Esto nos permitirá saber qué tenemos realmente antes de generar.

,count
categoria,
programasPregrado,871
reglamentos,57
postgrado,54
investigacion,23
vinculacion,19
calendarioAcademico,13
admisiones,6


### 8. Cuántos QA generaremos

No recomiendo generar la misma cantidad para todos.

Por ejemplo:
| Tipo           | QA |
| -------------- | -: |
| general        |  2 |
| requisitos     |  4 |
| procedimiento  |  5 |
| contacto       |  3 |
| objetivos      |  3 |
| perfil_ingreso |  3 |
| perfil_egreso  |  3 |
| fechas         |  3 |
| aranceles      |  3 |
| inscripción    |  4 |
| lista          |  3 |


In [10]:
QA_POR_TIPO = {

    "general": 3,

    "requisitos": 4,

    "procedimiento": 5,

    "contacto": 3,

    "objetivos": 3,

    "perfil_ingreso": 3,

    "perfil_egreso": 2,

    "fechas": 3,

    "aranceles": 3,

    "inscripcion": 4,

    "lista": 3
}

In [11]:
# Si el tipo no está definido:
def cantidad_qa(tipo):

    return QA_POR_TIPO.get(
        tipo,
        2
    )

### 9. Prompt base

Este punto es crítico.

Queremos evitar que el modelo "complete" información utilizando conocimiento externo.

In [12]:
SYSTEM_PROMPT = """
Eres un especialista en creación de datasets para entrenar
un asistente virtual institucional universitario.

Tu tarea es transformar fragmentos de información oficial
de una universidad en ejemplos de preguntas y respuestas.

REGLAS OBLIGATORIAS:

1. Utiliza EXCLUSIVAMENTE la información contenida en el fragmento.
2. No inventes información.
3. No completes datos que no estén explícitamente presentes.
4. No utilices conocimiento externo.
5. No agregues fechas, teléfonos, correos, requisitos, precios
   o procedimientos que no aparezcan en el texto.
6. Las respuestas deben ser claras, naturales y útiles.
7. Las preguntas deben parecer preguntas reales de estudiantes,
   profesores, investigadores, personal administrativo o visitantes.
8. Evita preguntas cuya respuesta sea simplemente "sí" o "no".
9. No menciones que la respuesta proviene de un fragmento.
10. Conserva exactamente nombres, fechas, cantidades, teléfonos,
    correos y nombres de programas cuando aparezcan.
11. Si el fragmento no contiene suficiente información para
    construir una pregunta útil, devuelve una lista vacía.

Devuelve ÚNICAMENTE JSON válido.
"""

### 10. Prompts especializados
### **Requisitos**

In [13]:
PROMPT_REQUISITOS = """
Genera preguntas que un estudiante podría hacer sobre los
requisitos de este proceso, programa o servicio.

Incluye diferentes formulaciones como:

- ¿Cuáles son los requisitos...?
- ¿Qué documentos necesito...?
- ¿Qué debo presentar...?
- ¿Qué se necesita para...?

No inventes requisitos.
"""

### **Procedimientos**

In [14]:
PROMPT_PROCEDIMIENTO = """
Genera preguntas prácticas sobre cómo realizar el procedimiento.

Incluye preguntas como:

- ¿Cómo puedo...?
- ¿Cuál es el procedimiento para...?
- ¿Qué pasos debo seguir...?
- ¿Qué debo hacer primero...?
- ¿Cómo realizo el proceso de...?

La respuesta debe explicar el procedimiento utilizando
únicamente la información disponible.
"""

## **Contactos**

In [15]:
PROMPT_CONTACTO = """
Genera preguntas relacionadas con cómo contactar a la dependencia
o responsable mencionado.

Ejemplos:

- ¿Cuál es el correo de...?
- ¿Cómo puedo contactar a...?
- ¿Cuál es el teléfono de...?
- ¿Dónde puedo comunicarme con...?

Conserva exactamente teléfonos y correos.
"""

In [16]:
PROMPT_FECHAS = """
Genera preguntas relacionadas con las fechas presentes en el texto.

Ejemplos:

- ¿Cuándo comienza...?
- ¿Cuál es la fecha de...?
- ¿Hasta cuándo puedo...?
- ¿Qué día se realizará...?

No inventes fechas.
"""

### **Aranceles**

In [17]:
PROMPT_ARANCELES = """
Genera preguntas relacionadas con costos, aranceles, precios
o montos presentes explícitamente en el texto. Siempre recuerda que los programas de pregrado son gratuitos, es decir no tienen aranceles de inscripcion ni de mensualidades.
Conserva exactamente los valores monetarios.
No hagas cálculos adicionales.
"""

### **Programas academicos**

In [18]:
PROMPT_PROGRAMAS = """
Genera preguntas relacionadas con el programa académico.

Puedes preguntar sobre:

- duración
- modalidad
- perfil
- objetivos
- estructura
- requisitos
- áreas de formación
- títulos
- créditos

Solo utiliza información presente en el texto.
"""

11. Seleccionar instrucciones según el tipo

In [19]:
PROMPTS_TIPO = {

    "requisitos":
        PROMPT_REQUISITOS,

    "procedimiento":
        PROMPT_PROCEDIMIENTO,

    "contacto":
        PROMPT_CONTACTO,

    "fechas":
        PROMPT_FECHAS,

    "aranceles":
        PROMPT_ARANCELES,

    "perfil_ingreso":
        PROMPT_PROGRAMAS,

    "perfil_egreso":
        PROMPT_PROGRAMAS,

    "objetivos":
        PROMPT_PROGRAMAS,

    "inscripcion":
        PROMPT_PROCEDIMIENTO,

    "lista":
        PROMPT_REQUISITOS,

    "general":
        ""
}

### 12. Función para generar QA

Vamos a pedir una estructura como:
{
  "qa": [
    {
      "question": "...",
      "answer": "..."
    }
  ]
}

In [20]:
def generar_qa(
    fragmento,
    modelo=GENERATION_MODEL
):

    texto = fragmento["texto"]

    tipo = fragmento.get(
        "tipo",
        "general"
    )

    cantidad = cantidad_qa(
        tipo
    )

    instrucciones = PROMPTS_TIPO.get(
        tipo,
        ""
    )

    user_prompt = f"""
Información institucional:

---
{texto}
---

Tipo de información:
{tipo}

{instrucciones}

Genera exactamente {cantidad} pares de
pregunta y respuesta.

Las preguntas deben ser diferentes entre sí.

No repitas simplemente la misma pregunta cambiando una palabra.

Devuelve:

{{
  "qa": [
    {{
      "question": "...",
      "answer": "..."
    }}
  ]
}}
"""

    try:

        response = client.chat.completions.create(

            model=modelo,

            temperature=0.2,

            response_format={
                "type": "json_object"
            },

            messages=[

                {
                    "role": "system",
                    "content": SYSTEM_PROMPT
                },

                {
                    "role": "user",
                    "content": user_prompt
                }
            ]
        )

        contenido = (
            response
            .choices[0]
            .message
            .content
        )

        return json.loads(
            contenido
        )

    except Exception as e:

        print(
            "Error generando QA:",
            e
        )

        return {
            "qa": []
        }

### 13. Probar con un solo fragmento

Antes de generar miles, hagamos esto para comprobar que las preguntas y respuestas tengan la calidad esperada

In [21]:
ejemplo = fragmentos[0]

print(
    ejemplo["texto"]
)

+58 (0243) 246-2211
[email protected]
Síguenos:
|
Portal Estudiantil
Intranet Docente
Postgrado
Oferta Académica de Postgrado
Estudios de Postgrado
UPT Aragua
Ofrecemos programas de postgrado o formación avanzada (PNFA) y diplomados orientados a la investigación, la innovación y el desarrollo tecnológico, en áreas estratégicas para la industria venezolana.
4
Programas PNFA
6
Diplomados
50+
Años de trayectoria
Especializaciones · Maestrías · Doctorados
Postgrados o Programas Nacionales de Formación Avanzada (PNFA)
Inscripción Abierta
Postgrado en Informática
Mención Desarrollo de Software
Especialización
Maestría
Doctorado
Forma investigadores y profesionales de alto nivel en el desarrollo de software, inteligencia artificial y sistemas de información.
1.5 – 4 años
Presencial
Próxima Apertura
Postgrado en Ingeniería Mecánica
Especialización
Maestría
Profundiza en el diseño mecánico avanzado, manufactura, materiales y mantenimiento de sistemas mecánicos industriales.
1.5 – 2 años
Presenc

In [22]:
resultado = generar_qa(
    ejemplo
)

resultado

{'qa': [{'question': '¿Cuáles son los requisitos generales para ingresar a los Programas Nacionales de Formación Avanzada (PNFA)?',
   'answer': 'Los requisitos generales son: título universitario de TSU, Licenciatura o Ingeniería; promedio mínimo de 14 puntos; carta de motivación y entrevista.'},
  {'question': '¿Qué documentos debo presentar para postularme a un programa de postgrado en la UPT Aragua?',
   'answer': 'Debes presentar tu título universitario de TSU, Licenciatura o Ingeniería, una carta de motivación y cumplir con un promedio mínimo de 14 puntos, además de realizar una entrevista.'},
  {'question': '¿Qué se necesita para inscribirse en un diplomado o programa PNFA en la UPT Aragua?',
   'answer': 'Para inscribirse en un programa PNFA se requiere título universitario de TSU, Licenciatura o Ingeniería, promedio mínimo de 14 puntos, carta de motivación y entrevista.'},
  {'question': '¿Qué requisitos académicos son indispensables para acceder a los postgrados ofrecidos por

### 14. Incorporar metadata

No queremos perder la procedencia.

In [23]:
def construir_registros_qa(
    fragmento,
    resultado
):

    registros = []

    for qa in resultado.get(
        "qa",
        []
    ):

        pregunta = qa.get(
            "question",
            ""
        ).strip()

        respuesta = qa.get(
            "answer",
            ""
        ).strip()

        if not pregunta or not respuesta:

            continue

        registro = {

            "question": pregunta,

            "answer": respuesta,

            "metadata": {

                "fragmento_id":
                    fragmento[
                        "fragmento_id"
                    ],

                "fuente":
                    fragmento[
                        "fuente"
                    ],

                "categoria":
                    fragmento[
                        "categoria"
                    ],

                "tipo":
                    fragmento[
                        "tipo"
                    ],

                "titulo":
                    fragmento[
                        "titulo_documento"
                    ],

                "seccion":
                    fragmento[
                        "seccion"
                    ],

                "url":
                    fragmento[
                        "url"
                    ],

                "documento":
                    fragmento[
                        "documento"
                    ]
            }
        }

        registros.append(
            registro
        )

    return registros

### 15. Procesar todo el corpus

Aquí recomiendo guardar periódicamente.

Si Colab se desconecta, no queremos perder 2 horas de trabajo.

In [24]:
QA_RAW_FILE = os.path.join(
    DATASET_DIR,
    "qa_generados.jsonl"
)

Antes de iniciar

In [25]:
if os.path.exists(QA_RAW_FILE):

    os.remove(
        QA_RAW_FILE
    )

**Procesamiento**

In [26]:
todos_qa = []

for i, fragmento in enumerate(
    tqdm(
        fragmentos,
        desc="Generando QA"
    )
):

    resultado = generar_qa(
        fragmento
    )

    registros = construir_registros_qa(
        fragmento,
        resultado
    )

    todos_qa.extend(
        registros
    )

    # Guardado incremental
    if (
        (i + 1) % 10 == 0
    ):

        with open(
            QA_RAW_FILE,
            "w",
            encoding="utf-8"
        ) as f:

            for item in todos_qa:

                f.write(
                    json.dumps(
                        item,
                        ensure_ascii=False
                    )
                    + "\n"
                )

    # Pausa pequeña
    time.sleep(0.2)


print(
    f"QA generados: {len(todos_qa)}"
)

Generando QA:   0%|          | 0/1043 [00:00<?, ?it/s]

QA generados: 3074


**Al finalizar**

In [27]:
with open(
    QA_RAW_FILE,
    "w",
    encoding="utf-8"
) as f:

    for item in todos_qa:

        f.write(
            json.dumps(
                item,
                ensure_ascii=False
            )
            + "\n"
        )

### 16. Eliminar duplicados

Primero normalizamos las preguntas.

In [28]:
def normalizar_pregunta(
    pregunta
):

    pregunta = pregunta.lower()

    pregunta = unidecode(
        pregunta
    )

    pregunta = re.sub(
        r"\s+",
        " ",
        pregunta
    )

    pregunta = re.sub(
        r"[¿?!.,;:]",
        "",
        pregunta
    )

    return pregunta.strip()

Importamos

In [30]:
get_ipython().system('pip install -q unidecode')
from unidecode import unidecode

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 7.2 MB/s eta 0:00:00


In [31]:
def eliminar_duplicados(
    registros
):

    vistos = set()

    resultado = []

    for item in registros:

        clave = (
            normalizar_pregunta(
                item["question"]
            )
        )

        if clave in vistos:

            continue

        vistos.add(
            clave
        )

        resultado.append(
            item
        )

    return resultado

**Ejecutamos**

In [32]:
qa_unicos = eliminar_duplicados(
    todos_qa
)

print(
    "Original:",
    len(todos_qa)
)

print(
    "Sin duplicados:",
    len(qa_unicos)
)

Original: 3074
Sin duplicados: 2956


17. Filtrar QA problemáticos

No queremos:

Pregunta: ¿Qué información contiene el texto?
Respuesta: El texto contiene información sobre...

Ya que eso no ensena nada al modelo

In [33]:
def es_qa_valido(item):

    pregunta = item["question"].strip()

    respuesta = item["answer"].strip()

    if len(pregunta.split()) < 4:

        return False

    if len(respuesta.split()) < 3:

        return False

    patrones_invalidos = [

        "según el texto",

        "según la información",

        "el fragmento",

        "no se especifica",

        "no se menciona"
    ]

    pregunta_lower = pregunta.lower()

    respuesta_lower = respuesta.lower()

    for patron in patrones_invalidos:

        if patron in pregunta_lower:

            return False

        if patron in respuesta_lower:

            return False

    return True

Aplicamos

In [34]:
qa_filtrados = [
    x
    for x in qa_unicos
    if es_qa_valido(x)
]

print(
    "QA después del filtro:",
    len(qa_filtrados)
)

QA después del filtro: 2821


### 18. Guardar dataset filtrado

In [35]:
QA_FILTERED_FILE = os.path.join(
    DATASET_DIR,
    "qa_filtrados.jsonl"
)

with open(
    QA_FILTERED_FILE,
    "w",
    encoding="utf-8"
) as f:

    for item in qa_filtrados:

        f.write(
            json.dumps(
                item,
                ensure_ascii=False
            )
            + "\n"
        )

### 19. Crear dataset de revisión

Esta etapa es muy importante.

No deberíamos entrenar inmediatamente.

In [36]:
df_qa = pd.DataFrame(
    qa_filtrados
)

df_qa["pregunta"] = (
    df_qa["question"]
)

df_qa["respuesta"] = (
    df_qa["answer"]
)

df_qa["categoria"] = (
    df_qa["metadata"]
    .apply(
        lambda x: x["categoria"]
    )
)

df_qa["tipo"] = (
    df_qa["metadata"]
    .apply(
        lambda x: x["tipo"]
    )
)

df_qa["fuente"] = (
    df_qa["metadata"]
    .apply(
        lambda x: x["fuente"]
    )
)

df_revision = df_qa[
    [
        "pregunta",
        "respuesta",
        "categoria",
        "tipo",
        "fuente"
    ]
]

Guardamos

In [37]:
REVISION_FILE = os.path.join(
    DATASET_DIR,
    "qa_revision.csv"
)

df_revision.to_csv(
    REVISION_FILE,
    index=False,
    encoding="utf-8-sig"
)

print(
    REVISION_FILE
)

/content/drive/MyDrive/UniversidadLLM/dataset/qa_revision.csv


### 20. Estadísticas del dataset

In [38]:
print(
    df_qa["categoria"]
    .value_counts()
)

categoria
programasPregrado      2282
reglamentos             199
postgrado               172
investigacion            67
vinculacion              60
admisiones               22
calendarioAcademico      19
Name: count, dtype: int64


In [39]:
print(
    df_qa["tipo"]
    .value_counts()
)

tipo
general           1755
objetivos          229
procedimiento      228
inscripcion        218
requisitos         178
lista               59
contacto            58
fechas              46
aranceles           40
perfil_egreso        6
perfil_ingreso       4
Name: count, dtype: int64


Esto es muy útil.

Podríamos descubrir, por ejemplo:

postgrado       1850
admisiones      1400
investigacion    650
vinculacion      500
general          900

Si vemos que admisiones tiene 1.400 y investigación solamente 80, podemos hacer oversampling controlado posteriormente.

21. Separar entrenamiento y validación

No hagas:

90% aleatorio
10% aleatorio

sin considerar el origen.

Podríamos terminar con:

Pregunta 1:
¿Cómo solicito admisión?

Pregunta 2:
¿Cuáles son los pasos para solicitar admisión?


una en train y otra en validation.

Eso produce una evaluación engañosa.

Primero agruparemos por fragmento_id.

In [40]:
from sklearn.model_selection import GroupShuffleSplit

Instalar si hace falta:

In [ ]:
!pip install -q scikit-learn

Crear grupos:

In [41]:
df_qa["fragmento_id"] = (
    df_qa["metadata"]
    .apply(
        lambda x: x["fragmento_id"]
    )
)

Separar

In [42]:
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        df_qa,
        groups=df_qa["fragmento_id"]
    )
)

train_df = df_qa.iloc[
    train_idx
].copy()

val_df = df_qa.iloc[
    val_idx
].copy()

print(
    "Train:",
    len(train_df)
)

print(
    "Validation:",
    len(val_df)
)

Train: 2545
Validation: 276


### 22. Convertir al formato de entrenamiento

Ahora transformamos:
{
  "question": "...",
  "answer": "..."
}

a

{
  "messages": [
    {
      "role": "user",
      "content": "..."
    },
    {
      "role": "assistant",
      "content": "..."
    }
  ]
}

In [43]:
def convertir_chatml(df):

    registros = []

    for _, row in df.iterrows():

        registros.append({

            "messages": [

                {
                    "role": "user",
                    "content":
                        row["question"]
                },

                {
                    "role": "assistant",
                    "content":
                        row["answer"]
                }

            ]
        })

    return registros

### 23. Crear train.jsonl

In [44]:
train_data = convertir_chatml(
    train_df
)

TRAIN_FILE = os.path.join(
    DATASET_DIR,
    "train.jsonl"
)

with open(
    TRAIN_FILE,
    "w",
    encoding="utf-8"
) as f:

    for item in train_data:

        f.write(
            json.dumps(
                item,
                ensure_ascii=False
            )
            + "\n"
        )

### 24. Crear validation.jsonl

In [45]:
val_data = convertir_chatml(
    val_df
)

VAL_FILE = os.path.join(
    DATASET_DIR,
    "validation.jsonl"
)

with open(
    VAL_FILE,
    "w",
    encoding="utf-8"
) as f:

    for item in val_data:

        f.write(
            json.dumps(
                item,
                ensure_ascii=False
            )
            + "\n"
        )

### 25. Verificar un ejemplo

In [46]:
train_data[0]

{'messages': [{'role': 'user',
   'content': '¿Cuáles son los requisitos generales para ingresar a los Programas Nacionales de Formación Avanzada (PNFA)?'},
  {'role': 'assistant',
   'content': 'Los requisitos generales para ingresar a los PNFA son: tener título universitario de TSU, Licenciatura o Ingeniería, un promedio mínimo de 14 puntos, presentar una carta de motivación y realizar una entrevista.'}]}

Debe producir algo parecido a:

{
  "messages": [
    {
      "role": "user",
      "content": "¿Cuáles son los requisitos para ingresar al programa?"
    },
    {
      "role": "assistant",
      "content": "Para ingresar al programa se requiere..."
    }
  ]
}

### 26. Pero hay una mejora que recomiendo hacer antes de entrenar

No quiero que nuestro dataset termine siendo solamente:

pregunta factual
respuesta factual

Necesitamos diversidad de intención.

Por ejemplo, para:

Ingeniería Informática tiene una duración de cinco años.

No deberíamos generar únicamente:

¿Cuánto dura Ingeniería Informática?

Podemos generar:

¿Cuántos años dura Ingeniería Informática?

¿Cuál es la duración de Ingeniería Informática?

¿Qué duración tiene el programa de Ingeniería Informática?

Si quiero estudiar Ingeniería Informática, ¿cuánto tiempo dura la carrera?

Esto enseña al modelo a reconocer diferentes formas de preguntar.

### 27. Y necesitamos otro tipo de ejemplos

Por ejemplo, para requisitos:

¿Cuáles son los requisitos para inscribirme?

¿Qué documentos necesito presentar?

¿Qué debo llevar para completar la inscripción?

Quiero inscribirme, ¿qué requisitos debo cumplir?

Pero sin cambiar la respuesta factual.

### 28. También necesitamos preguntas de contexto

Por ejemplo:

¿Qué programas de postgrado ofrece la universidad?

¿Cuáles son los programas de postgrado relacionados con informática?

¿Qué opciones de formación avanzada existen?

¿Qué diplomados ofrece actualmente la universidad?

Esto hará que el modelo sea más útil conversacionalmente.

### 29. Y preguntas negativas/limitaciones

Hay que tener cuidado aquí.

Si el documento dice:

La inscripción se realiza únicamente de manera presencial.

Podemos generar:

¿Puedo realizar la inscripción en línea?

Respuesta:

No. Según la información institucional disponible, la inscripción se realiza únicamente de manera presencial.

Esto es muy importante para reducir alucinaciones, porque enseñamos al modelo a responder correctamente cuando la pregunta tiene una premisa diferente.

### 30. Lo que tendremos al final

Nuestro pipeline completo será:

                 DOCUMENTOS
                     │
                     ▼
             Extracción / Scraping
                     │
                     ▼
             Limpieza y normalización
                     │
                     ▼
          Fragmentación semántica
                     │
                     ▼
          ┌──────────────────────┐
          │ Fragmentos           │
          │ institucionales      │
          └──────────┬───────────┘
                     │
                     ▼
             Generación QA
                     │
                     ▼
          Eliminación duplicados
                     │
                     ▼
             Filtros de calidad
                     │
                     ▼
             Revisión humana
                     │
                     ▼
           Dataset equilibrado
                     │
                     ▼
              Train / Validation
                     │
                     ▼
                  QLoRA
                     │
                     ▼
              Modelo 7B/8B

### Una advertencia importante

No entrenaría todavía el modelo con el train.jsonl recién generado. Primero revisaría manualmente una muestra de unas 300–500 preguntas. En un proyecto institucional como el tuyo, es preferible tener 5.000 QA excelentes que 20.000 QA mediocremente generados.

Además, en la siguiente iteración haría una segunda pasada automática sobre el dataset: un QA Judge, que tome pregunta + respuesta + fragmento original y determine si la respuesta realmente está soportada por la fuente. Así podemos eliminar automáticamente respuestas inventadas, respuestas incompletas, preguntas redundantes y QA de baja calidad antes del QLoRA. Esa capa es probablemente la que más valor va a aportar a la calidad final de tu primer modelo institucional.